# Aproximación de Funciones con Redes Neuronales

**Aprendizaje Profundo - UNSAM**

En este notebook exploraremos:
1. Cómo las redes neuronales aproximan funciones
2. Efecto del número de neuronas en capas ocultas
3. Comparación entre redes con 1 y 2 capas ocultas
4. Visualización del aprendizaje en tiempo real

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output

# Configuración
torch.manual_seed(42)
np.random.seed(42)

## 1. Generación de Datos

Creamos una función escalonada con tres niveles:
- Y = 1.0 para X ≤ -10
- Y = 0.5 para -10 < X < 10
- Y = 0.0 para X ≥ 10

In [ ]:
# Generate synthetic data for X ranging from -30 to 29 with a step size of 1
X = torch.arange(-30, 30, 1).view(-1, 1).type(torch.FloatTensor)

# Initialize an empty tensor Y to store the labels (target values)
Y = torch.zeros(X.shape[0])

# Assign label 1.0 to elements in Y where the corresponding X value is less than or equal to -10
Y[X[:, 0] <= -10] = 1.0

# Assign label 0.5 to elements in Y where the corresponding X value falls between -10 and 10 (exclusive)
Y[(X[:, 0] > -10) & (X[:, 0] < 10)] = 0.5

# Assign label 0 to elements in Y where the corresponding X value is greater than 10
Y[X[:, 0] > 10] = 0

# Reshape Y to match X dimensions
Y = Y.view(-1, 1)

print(f"Forma de X: {X.shape}")
print(f"Forma de Y: {Y.shape}")
print(f"Rango de X: [{X.min():.1f}, {X.max():.1f}]")
print(f"Valores únicos de Y: {torch.unique(Y)}")

In [ ]:
# Visualizar los datos
plt.figure(figsize=(10, 5))
plt.scatter(X.numpy(), Y.numpy(), alpha=0.6, s=50, edgecolors='k')
plt.xlabel('X', fontsize=12)
plt.ylabel('Y', fontsize=12)
plt.title('Función Escalonada - Datos de Entrenamiento', fontsize=14)
plt.grid(True, alpha=0.3)
plt.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
plt.axhline(y=0.5, color='k', linestyle='--', linewidth=0.5, alpha=0.5)
plt.axhline(y=1.0, color='k', linestyle='--', linewidth=0.5, alpha=0.5)
plt.axvline(x=-10, color='r', linestyle='--', linewidth=1, alpha=0.5, label='Transiciones')
plt.axvline(x=10, color='r', linestyle='--', linewidth=1, alpha=0.5)
plt.legend()
plt.show()

## 2. Red Neuronal con 1 Capa Oculta

Definimos una red neuronal flexible donde podemos configurar el número de neuronas en la capa oculta.

In [ ]:
class OneLayerNet(nn.Module):
    """Red neuronal con una capa oculta."""
    
    def __init__(self, input_size, hidden_size, output_size):
        super(OneLayerNet, self).__init__()
        self.hidden = nn.Linear(input_size, hidden_size)
        self.output = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        x = torch.relu(self.hidden(x))
        x = self.output(x)
        return x

# Crear modelo
hidden_neurons = 10  # CONFIGURABLE: Cambia este valor para experimentar
model_1layer = OneLayerNet(input_size=1, hidden_size=hidden_neurons, output_size=1)

print(f"Modelo con {hidden_neurons} neuronas en la capa oculta:")
print(model_1layer)
print(f"\nNúmero total de parámetros: {sum(p.numel() for p in model_1layer.parameters())}")

### Función de Entrenamiento con Visualización

In [ ]:
def train_and_visualize(model, X, Y, epochs=1000, lr=0.01, plot_every=100):
    """
    Entrena el modelo y visualiza el progreso.
    
    Args:
        model: Modelo de red neuronal
        X: Datos de entrada
        Y: Etiquetas objetivo
        epochs: Número de épocas de entrenamiento
        lr: Tasa de aprendizaje
        plot_every: Frecuencia de actualización del gráfico
    """
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    losses = []
    
    # Crear puntos para predicción suave
    X_plot = torch.linspace(X.min(), X.max(), 200).view(-1, 1)
    
    for epoch in range(epochs):
        # Forward pass
        model.train()
        outputs = model(X)
        loss = criterion(outputs, Y)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        losses.append(loss.item())
        
        # Visualizar progreso
        if (epoch + 1) % plot_every == 0 or epoch == 0:
            clear_output(wait=True)
            
            model.eval()
            with torch.no_grad():
                Y_pred = model(X_plot)
            
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
            
            # Gráfico de la función
            ax1.scatter(X.numpy(), Y.numpy(), alpha=0.6, s=30, 
                       edgecolors='k', label='Datos reales', zorder=3)
            ax1.plot(X_plot.numpy(), Y_pred.numpy(), 'r-', linewidth=2, 
                    label='Predicción', zorder=2)
            ax1.set_xlabel('X', fontsize=12)
            ax1.set_ylabel('Y', fontsize=12)
            ax1.set_title(f'Época {epoch + 1}/{epochs} - Pérdida: {loss.item():.4f}', 
                         fontsize=14)
            ax1.grid(True, alpha=0.3)
            ax1.legend()
            ax1.axvline(x=-10, color='gray', linestyle='--', linewidth=1, alpha=0.3)
            ax1.axvline(x=10, color='gray', linestyle='--', linewidth=1, alpha=0.3)
            
            # Gráfico de la pérdida
            ax2.plot(losses, 'b-', linewidth=2)
            ax2.set_xlabel('Época', fontsize=12)
            ax2.set_ylabel('Pérdida (MSE)', fontsize=12)
            ax2.set_title('Curva de Aprendizaje', fontsize=14)
            ax2.grid(True, alpha=0.3)
            ax2.set_yscale('log')
            
            plt.tight_layout()
            plt.show()
            
            print(f"Época [{epoch + 1}/{epochs}], Pérdida: {loss.item():.6f}")
    
    return losses

### Entrenar el Modelo con 1 Capa Oculta

**Instrucciones**: Ejecuta esta celda para entrenar el modelo. Observa cómo la red aprende a aproximar la función escalonada.

In [ ]:
# Entrenar el modelo
print(f"Entrenando red con 1 capa oculta ({hidden_neurons} neuronas)...\n")
losses_1layer = train_and_visualize(model_1layer, X, Y, epochs=2000, lr=0.01, plot_every=200)

### Experimenta con Diferentes Números de Neuronas

**Actividad**: Vuelve a la celda donde definimos `hidden_neurons` y prueba con diferentes valores:
- `hidden_neurons = 5` (pocas neuronas)
- `hidden_neurons = 10` (moderado)
- `hidden_neurons = 20` (muchas neuronas)
- `hidden_neurons = 50` (red grande)

Observa cómo cambia la aproximación de la función.

## 3. Red Neuronal con 2 Capas Ocultas

Ahora definimos una red con dos capas ocultas para comparar el rendimiento.

In [ ]:
class TwoLayerNet(nn.Module):
    """Red neuronal con dos capas ocultas."""
    
    def __init__(self, input_size, hidden1_size, hidden2_size, output_size):
        super(TwoLayerNet, self).__init__()
        self.hidden1 = nn.Linear(input_size, hidden1_size)
        self.hidden2 = nn.Linear(hidden1_size, hidden2_size)
        self.output = nn.Linear(hidden2_size, output_size)
        
    def forward(self, x):
        x = torch.relu(self.hidden1(x))
        x = torch.relu(self.hidden2(x))
        x = self.output(x)
        return x

# Crear modelo
hidden1_neurons = 8  # CONFIGURABLE: Primera capa oculta
hidden2_neurons = 6  # CONFIGURABLE: Segunda capa oculta
model_2layer = TwoLayerNet(input_size=1, 
                           hidden1_size=hidden1_neurons, 
                           hidden2_size=hidden2_neurons, 
                           output_size=1)

print(f"Modelo con 2 capas ocultas ({hidden1_neurons}, {hidden2_neurons} neuronas):")
print(model_2layer)
print(f"\nNúmero total de parámetros: {sum(p.numel() for p in model_2layer.parameters())}")

### Entrenar el Modelo con 2 Capas Ocultas

In [ ]:
# Entrenar el modelo
print(f"Entrenando red con 2 capas ocultas ({hidden1_neurons}, {hidden2_neurons} neuronas)...\n")
losses_2layer = train_and_visualize(model_2layer, X, Y, epochs=2000, lr=0.01, plot_every=200)

## 4. Comparación Sistemática de Arquitecturas

Ahora comparemos múltiples arquitecturas de forma sistemática.

In [ ]:
def train_model_silent(model, X, Y, epochs=2000, lr=0.01):
    """Entrenar modelo sin visualización para comparaciones rápidas."""
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    losses = []
    
    for epoch in range(epochs):
        model.train()
        outputs = model(X)
        loss = criterion(outputs, Y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        losses.append(loss.item())
    
    return losses

# Definir diferentes configuraciones
configs = [
    {'name': '1 capa - 5 neuronas', 'layers': [5], 'params': None},
    {'name': '1 capa - 10 neuronas', 'layers': [10], 'params': None},
    {'name': '1 capa - 20 neuronas', 'layers': [20], 'params': None},
    {'name': '2 capas - [8, 6]', 'layers': [8, 6], 'params': None},
    {'name': '2 capas - [10, 8]', 'layers': [10, 8], 'params': None},
    {'name': '2 capas - [15, 10]', 'layers': [15, 10], 'params': None},
]

results = {}

print("Entrenando múltiples arquitecturas...\n")

for config in configs:
    print(f"Entrenando: {config['name']}")
    
    # Crear modelo según configuración
    if len(config['layers']) == 1:
        model = OneLayerNet(1, config['layers'][0], 1)
    else:
        model = TwoLayerNet(1, config['layers'][0], config['layers'][1], 1)
    
    # Contar parámetros
    n_params = sum(p.numel() for p in model.parameters())
    config['params'] = n_params
    
    # Entrenar
    losses = train_model_silent(model, X, Y, epochs=2000, lr=0.01)
    
    # Obtener predicción final
    model.eval()
    X_plot = torch.linspace(X.min(), X.max(), 200).view(-1, 1)
    with torch.no_grad():
        Y_pred = model(X_plot)
    
    results[config['name']] = {
        'model': model,
        'losses': losses,
        'final_loss': losses[-1],
        'params': n_params,
        'prediction': (X_plot, Y_pred)
    }
    
    print(f"  - Parámetros: {n_params}")
    print(f"  - Pérdida final: {losses[-1]:.6f}\n")

print("¡Entrenamiento completado!")

### Visualización Comparativa de Todas las Arquitecturas

In [ ]:
# Visualizar todas las predicciones
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

for idx, (name, result) in enumerate(results.items()):
    X_plot, Y_pred = result['prediction']
    
    axes[idx].scatter(X.numpy(), Y.numpy(), alpha=0.4, s=30, 
                     edgecolors='k', label='Datos reales', zorder=3)
    axes[idx].plot(X_plot.numpy(), Y_pred.numpy(), 'r-', linewidth=2, 
                  label='Predicción', zorder=2)
    axes[idx].set_xlabel('X')
    axes[idx].set_ylabel('Y')
    axes[idx].set_title(f'{name}\nParams: {result["params"]}, '
                       f'Pérdida: {result["final_loss"]:.4f}')
    axes[idx].grid(True, alpha=0.3)
    axes[idx].legend(loc='upper right', fontsize=8)
    axes[idx].axvline(x=-10, color='gray', linestyle='--', linewidth=1, alpha=0.3)
    axes[idx].axvline(x=10, color='gray', linestyle='--', linewidth=1, alpha=0.3)

plt.tight_layout()
plt.show()

### Comparación de Curvas de Aprendizaje

In [ ]:
# Comparar curvas de aprendizaje
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
for name, result in results.items():
    plt.plot(result['losses'], label=name, linewidth=2, alpha=0.8)
plt.xlabel('Época', fontsize=12)
plt.ylabel('Pérdida (MSE)', fontsize=12)
plt.title('Curvas de Aprendizaje - Todas las Arquitecturas', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')

plt.subplot(1, 2, 2)
# Gráfico de barras: Parámetros vs Pérdida Final
names = list(results.keys())
params = [results[name]['params'] for name in names]
final_losses = [results[name]['final_loss'] for name in names]

colors = ['blue']*3 + ['red']*3  # Azul para 1 capa, rojo para 2 capas
bars = plt.bar(range(len(names)), final_losses, color=colors, alpha=0.7, edgecolor='k')
plt.xticks(range(len(names)), [f"{p}\nparams" for p in params], rotation=0)
plt.ylabel('Pérdida Final (MSE)', fontsize=12)
plt.title('Pérdida Final vs Número de Parámetros', fontsize=14)
plt.grid(True, alpha=0.3, axis='y')

# Leyenda
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='blue', alpha=0.7, label='1 Capa Oculta'),
                   Patch(facecolor='red', alpha=0.7, label='2 Capas Ocultas')]
plt.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.show()

### Tabla Comparativa de Resultados

In [ ]:
import pandas as pd

# Crear tabla comparativa
comparison_data = []
for name, result in results.items():
    comparison_data.append({
        'Arquitectura': name,
        'Parámetros': result['params'],
        'Pérdida Final': f"{result['final_loss']:.6f}",
        'Pérdida Inicial': f"{result['losses'][0]:.6f}",
        'Mejora (%)': f"{((result['losses'][0] - result['final_loss']) / result['losses'][0] * 100):.2f}%"
    })

df_comparison = pd.DataFrame(comparison_data)
print("\n" + "="*80)
print("TABLA COMPARATIVA DE ARQUITECTURAS")
print("="*80)
print(df_comparison.to_string(index=False))
print("="*80)

## 5. Análisis y Conclusiones

### Preguntas para Reflexionar:

1. **¿Cómo afecta el número de neuronas a la calidad de la aproximación?**
   - Con pocas neuronas (ej: 5), la red tiene dificultades para capturar las transiciones bruscas
   - Con más neuronas, la aproximación mejora pero puede haber sobreajuste

2. **¿Qué diferencia observas entre 1 y 2 capas ocultas?**
   - Las redes con 2 capas pueden aprender representaciones más complejas
   - Con menos parámetros totales, pueden lograr mejor aproximación

3. **¿Existe un número "óptimo" de parámetros?**
   - Hay un balance entre capacidad de la red y generalización
   - Más parámetros no siempre significa mejor rendimiento

4. **¿Por qué es difícil aproximar una función escalonada?**
   - Las transiciones bruscas requieren muchas neuronas
   - Las funciones de activación suaves (ReLU, sigmoid) deben combinar para crear discontinuidades

### Observaciones Clave:

- **Profundidad vs Anchura**: Las redes profundas (2 capas) pueden ser más eficientes que las anchas (1 capa con muchas neuronas)
- **Capacidad de Representación**: Una red con más parámetros tiene mayor capacidad, pero también mayor riesgo de sobreajuste
- **Transiciones Suaves**: Las redes neuronales tienden a suavizar transiciones bruscas debido a sus funciones de activación

## 6. Ejercicios Adicionales

1. **Experimenta con diferentes funciones de activación**: Modifica las clases `OneLayerNet` y `TwoLayerNet` para usar `tanh` o `sigmoid` en lugar de `ReLU`. ¿Cómo cambia la aproximación?

2. **Prueba con diferentes tasas de aprendizaje**: Cambia el parámetro `lr` en las funciones de entrenamiento. ¿Qué sucede con `lr=0.001` vs `lr=0.1`?

3. **Añade una tercera capa oculta**: Crea una clase `ThreeLayerNet` y compárala con las anteriores.

4. **Modifica la función objetivo**: Cambia los valores de Y para crear una función diferente (ej: sinusoidal, exponencial) y observa cómo se adaptan las redes.

5. **Implementa Early Stopping**: Modifica la función de entrenamiento para detener cuando la pérdida deje de mejorar.

6. **Visualiza las activaciones**: Añade código para visualizar las activaciones de las capas ocultas durante el entrenamiento.

In [ ]:
# Espacio para experimentación

## Resumen

En este notebook aprendimos:
- Cómo las redes neuronales aproximan funciones complejas
- El impacto del número de neuronas en la calidad de la aproximación
- La diferencia entre redes con 1 y 2 capas ocultas
- Cómo comparar sistemáticamente diferentes arquitecturas
- El balance entre complejidad del modelo y rendimiento

### Conceptos Clave:
- **Teorema de Aproximación Universal**: Una red neuronal con una capa oculta puede aproximar cualquier función continua
- **Profundidad vs Anchura**: Las redes profundas pueden ser más eficientes que las anchas
- **Capacidad del Modelo**: Más parámetros = mayor capacidad, pero cuidado con el sobreajuste